In [ ]:
from typing import Iterable

import MeCab
import pandas as pd
from matplotlib import pyplot as plt
import japanize_matplotlib  # 日本語を図で表示するのに必要
import seaborn as sns

In [ ]:
tagger = MeCab.Tagger()

In [ ]:
yume = pd.read_csv('./data/textmining/yumejuya.tsv', sep='\t')

In [ ]:
yume.head()

In [ ]:
# pd.Seriesはstrアクセサを使うと文字列関連のメソッドを列に適用できる
yume['content'].str.len().hist(bins=25)
plt.xlabel('Paragraph Length')
plt.ylabel('Frequency')
plt.title('Histogram of paragraph length')

In [ ]:
sns.boxplot(x=yume['section_id'], y=yume['content'].str.len())
plt.ylabel('Length')

In [ ]:
# 分析のため各話ごとに文章を結合する
# groupbyを使ってセクションごとに文を結合

sections = yume.groupby('section_id')['content'].agg(''.join).reset_index()

In [ ]:
# 形状を確認
sections.shape

In [ ]:
sections

In [ ]:
sections['content'].str.len().plot(kind='bar')
plt.xticks(range(0, 10), sections['section_id'])
plt.xlabel('section_id')

In [ ]:
import functions as fn  # 自作関数
from collections import Counter

In [ ]:
def filter_tokens(data: list[fn.Token], pos: Iterable[str] | None = None) -> list[str]:
    # 特定の品詞の語のみ抜き出す
    if not pos:
        return data

    pos_ = set(pos)
    return [x.surface for x in data if x.pos in pos_]


def count_tokens(data: list[fn.Token], pos: Iterable[str] | None = None) -> pd.Series:
    # 語を表層形に基づいて数える
    return pd.Series(Counter(filter_tokens(data, pos)))


def count_tokens_df(data: pd.Series, pos: Iterable[str] | None = None) -> pd.DataFrame:
    # 行ごとに語を集計した結果をデータフレームで結合して返す
    return data.apply(fn.tokenize).apply(count_tokens, pos=pos).fillna(0).astype(int).T.sort_index()


In [ ]:
# extract_tokens(fn.tokenize(sections['content'][0]), pos=['名詞'])
# sections['content'].apply(fn.tokenize).apply(count_tokens, pos=['名詞']).fillna(0).astype(int)
count_tokens_df(sections['content'], pos=['名詞'])

In [ ]:
count_noun = count_tokens_df(sections['content'], pos=['名詞'])

In [ ]:
freq_noun = count_noun.sum(axis=1)

In [ ]:
plt.scatter(range(len(freq_noun)), freq_noun.sort_values(ascending=False).values, s=1)
plt.title('Distribution of noun frequency')
plt.xlabel('Rank')
plt.ylabel('Frequency')

In [ ]:
plt.scatter(range(len(freq_noun)), freq_noun.sort_values(ascending=False).values, s=1)
plt.xscale('log')
plt.yscale('log')
plt.title('Distribution of noun frequency')
plt.xlabel('Rank')
plt.ylabel('Frequency')

In [ ]:
import numpy as np

In [ ]:
# cf. Noraml Distribution
# mu = 0, sigma = 1
dist = np.random.normal(0, 1, 10000)
plt.hist(dist, bins=100)
plt.title('Normal distribution (n=10000)')
plt.show()

plt.scatter(range(10000), sorted(dist, reverse=True), s=1)
plt.title('Normal distribution (n=10000)')
plt.xlabel('Rank')
plt.ylabel('Value')

In [ ]:
# cf. Poissson Distribution
# lambda = 1
dist = np.random.poisson(1, 10000)

plt.hist(dist, bins=7)
plt.title('Poisson distribution (n=10000, lambda=1)')
plt.show()

plt.scatter(range(10000), sorted(dist, reverse=True), s=1)
plt.title('Poisson distribution (n=10000, lambda=1)')
plt.xlabel('Rank')
plt.ylabel('Value')

In [ ]:
freq_noun.sort_values(ascending=False).head(30)

In [ ]:
freq_noun.sort_values().tail(30).plot(kind='barh')

In [ ]:
stopwords = set(['よう', '上', '中',  'もの', 'の', 'それ', '一', '事', '何','ん', 'どこ'])

In [ ]:
freq_noun[[x not in stopwords for x in freq_noun.index]].sort_values().tail(30).plot(kind='barh')
plt.title('Top 30 nouns')
plt.ylabel('Frequency')

# ※順位タイの表示の仕方が違うためR版と若干結果が異なる

### TF-IDF

$$
TF_{i,j} = \frac{n_{i,j}}{\sum{_{k}n_{k,j}}} = \frac{文書 d_j における単語 t_i  の頻度}{文書d中の総単語数}
$$

$$
IDF{i,j} = \log{\frac{|D|}{|{d:d \ni t_i}|}} = \log{(1 / \frac{ 単語t_iを含む文書数}{ 総文書数})}
$$

$$
TFIDF_{i,j} = TF_{i,j} \times IDF_{i,j}
$$


In [ ]:
def tf(df: pd.DataFrame) -> pd.DataFrame:
    return (df / df.sum()).T


def idf(df: pd.DataFrame) -> pd.Series:
    doc_sums = (df > 0).sum(axis=1) + 1
    return np.log2(len(df.columns) / doc_sums)

In [ ]:
count_noun.head()

In [ ]:
tfidf = (tf(count_noun) * idf(count_noun)).T

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, ax in enumerate(axes.flat):
    d = tfidf.iloc[:, i].sort_values().tail(10)
    ax.barh(d.index, d.values)
    # ax.set_title(f"Plot {i}")

plt.tight_layout()
plt.show()